# 面试问题：LLM 做 SFT 时，Chat Template、Loss Mask 与 Packing 怎样正确实现？

**一句话回答**：先把 role/content/tool 边界定义成版本化 chat protocol，再用同一 tokenizer/template 编码训练和推理；causal LM 的输入/标签错一位，通常只对 assistant 目标 token 计算 loss，system/user/padding 设 ignore；packing 可共享一个物理序列，但 attention 和 position 必须阻断不同会话，且不能截断到破坏 role/tool pair。

本 Notebook 用简化词级 tokenizer 与 PyTorch 基础参数手写模板、assistant-only mask、shift、block causal mask、packing 和最小 `TinyLM.forward`。重点不是训练一个有用语言模型，而是建立可单测的数据管线 oracle：任意模板、截断或 packing 优化，都必须证明监督 token 数、角色边界和注意力隔离没有改变。

In [ ]:
from dataclasses import dataclass
import hashlib, json, math, re
import numpy as np
import torch
from torch import nn

SEED112=11201; torch.manual_seed(SEED112)
SPECIAL112={"<system>":0,"<user>":1,"<assistant>":2,"<eos>":3,"<pad>":4,"<unk>":5}
assert len(SPECIAL112)==6 and len(set(SPECIAL112.values()))==6
assert SEED112==11201
assert SPECIAL112["<pad>"]!=SPECIAL112["<eos>"]

## 1. 对话数据合同先于 tokenization

每条会话有稳定 ID、按序 messages、允许 role 和来源；assistant 为空、连续两个 tool result、缺失 tool call 等都应在编码前拒绝。数据按用户/文档族切分，避免同一问答改写进入 train/test。

In [ ]:
@dataclass(frozen=True)
class Message112: role:str; content:str
def validate_conversation112(messages):
    if not messages or any(m.role not in {"system","user","assistant"} or not m.content for m in messages): return False
    if messages[0].role not in {"system","user"}: return False
    return all(not (a.role==b.role=="assistant") for a,b in zip(messages,messages[1:]))
conv112=[Message112("system","help safely"),Message112("user","two plus two"),Message112("assistant","four")]
assert validate_conversation112(conv112)
assert not validate_conversation112([Message112("assistant","answer")])
assert not validate_conversation112([Message112("user","q"),Message112("assistant","a"),Message112("assistant","b")])

## 2. 词表只能在 train 拟合，special token ID 是接口

教学 tokenizer 以空格切词；真实模型必须使用 checkpoint 对应 tokenizer、normalizer 和 special tokens。验证/test 的新词映射 `<unk>`，不能为了降低 OOV 偷看评测集扩词表。chat template 变化会改变 token 序列，也等同模型接口变化。

In [ ]:
def words112(text): return text.lower().split()
train_text112=" ".join(m.content for m in conv112); vocab112=dict(SPECIAL112)
for w in sorted(set(words112(train_text112))): vocab112.setdefault(w,len(vocab112))
def encode_text112(text): return [vocab112.get(w,SPECIAL112["<unk>"]) for w in words112(text)]
assert encode_text112("four")[0]!=SPECIAL112["<unk>"]
assert encode_text112("unseenword")==[SPECIAL112["<unk>"]]
assert all(vocab112[k]==v for k,v in SPECIAL112.items())

## 3. Template 同时生成 token、role 边界和 loss mask

每条 message 编码为 role token + content + EOS。这里只把 assistant content 与其 EOS 标为训练目标；system/user token 的 mask 为 0。某些任务会训练 assistant role token 或 tool call，必须明确合同，而不是事后猜 mask。

In [ ]:
def render112(messages):
    ids=[]; mask=[]; roles=[]
    for m in messages:
        part=[SPECIAL112[f"<{m.role}>"]]+encode_text112(m.content)+[SPECIAL112["<eos>"]]; target=[0]+[int(m.role=="assistant")]*(len(part)-1)
        ids.extend(part); mask.extend(target); roles.extend([m.role]*len(part))
    return np.array(ids,np.int64),np.array(mask,np.int64),roles
ids112,mask112,roles112=render112(conv112)
assert len(ids112)==len(mask112)==len(roles112)
assert mask112.sum()==2  # "four" 与 assistant EOS
assert all(mask112[i]==0 for i,r in enumerate(roles112) if r!="assistant")

## 4. Causal LM 标签要左移，mask 对齐预测目标

`logits[t]` 预测 `token[t+1]`，所以输入去末 token、label 去首 token，监督 mask 也去首位置。padding 或非 assistant 目标写成 `ignore_index=-100`。off-by-one 会让模型学习复制当前 token，是常见隐蔽 bug。

In [ ]:
IGNORE112=-100
def causal_example112(ids,target_mask):
    x=ids[:-1].copy(); labels=ids[1:].copy(); shifted=target_mask[1:]; labels[shifted==0]=IGNORE112; return x,labels
input112,labels112=causal_example112(ids112,mask112)
supervised112=np.where(labels112!=IGNORE112)[0]
assert len(input112)==len(ids112)-1 and len(labels112)==len(input112)
assert len(supervised112)==2
assert labels112[supervised112[-1]]==SPECIAL112["<eos>"]

## 5. Packing 不能让会话跨边界注意

多个短样本拼进一个物理序列提高利用率，但每个 token 只能看同会话的过去。构造 `same_segment & causal` 的 block-diagonal mask，并为每段重置 position ID；若实现选择允许跨样本注意，必须证明不会引入训练/推理不一致或隐私泄漏。

In [ ]:
conv_b112=[Message112("user","two plus two"),Message112("assistant","four")]; ids_b112,mask_b112,_=render112(conv_b112); packed_ids112=np.r_[ids112,ids_b112]; segments112=np.r_[np.zeros(len(ids112),int),np.ones(len(ids_b112),int)]
pos112=np.empty(len(packed_ids112),int)
for seg in np.unique(segments112): pos112[segments112==seg]=np.arange(np.sum(segments112==seg))
idx112=np.arange(len(packed_ids112)); block_causal112=(segments112[:,None]==segments112[None,:])&(idx112[:,None]>=idx112[None,:])
assert block_causal112.shape==(len(packed_ids112),len(packed_ids112))
assert not block_causal112[len(ids112),len(ids112)-1]
assert pos112[len(ids112)]==0 and np.all(np.diag(block_causal112))

## 6. Truncation 要按 message/tool 边界

最好先过滤超长样本或按完整 turn 切片；不能留下 assistant role 却删掉答案，也不能把 tool call 与 result 分开。若目标 token 全被截掉，这条样本对 SFT 没有训练信号，应丢弃并计数，而不是产生除零 loss。

In [ ]:
def fit_complete_messages112(messages,max_tokens):
    kept=[]; used=0
    for m in messages:
        cost=2+len(encode_text112(m.content))
        if used+cost>max_tokens: break
        kept.append(m); used+=cost
    ids,mask,_=render112(kept) if kept else (np.array([],int),np.array([],int),[])
    return kept,ids,mask
kept112,tids112,tmask112=fit_complete_messages112(conv112,len(ids112)-1)
assert len(kept112)==2
assert tmask112.sum()==0
assert len(tids112)<len(ids112)

## 7. 用最小 `nn.Module` 验证 loss mask 真能训练

下面实现 bigram causal LM：当前位置 embedding 经矩阵产生下一 token logits。它不是完整 Transformer，但能独立验证 template、shift、ignore 和梯度合同。交叉熵用 `logsumexp` 手写，只在 assistant 目标位置求均值。

In [ ]:
class TinyLM112(nn.Module):
    def __init__(self,vocab_size,dim=16):
        super().__init__(); self.embedding=nn.Parameter(torch.randn(vocab_size,dim)*.1); self.proj=nn.Parameter(torch.randn(dim,vocab_size)*.1); self.bias=nn.Parameter(torch.zeros(vocab_size))
    def forward(self,input_ids): return self.embedding[input_ids]@self.proj+self.bias
def masked_ce112(logits,labels):
    valid=labels!=IGNORE112
    if not valid.any(): raise ValueError("no_supervised_tokens")
    z=logits[valid]; y=labels[valid]; return (torch.logsumexp(z,dim=-1)-z[torch.arange(len(y)),y]).mean()
model112=TinyLM112(len(vocab112)); tx112=torch.tensor(input112); ty112=torch.tensor(labels112); losses112=[]
for _ in range(100):
    loss=masked_ce112(model112(tx112),ty112); losses112.append(float(loss.detach())); loss.backward()
    with torch.no_grad():
        for p in model112.parameters(): p-=.3*p.grad; p.grad=None
assert losses112[-1]<losses112[0]*.1
assert model112(tx112).shape==(len(input112),len(vocab112))
assert all(torch.isfinite(p).all() for p in model112.parameters())

## 8. 数据审计与发布合同

保存原始数据快照、去重规则、split key、tokenizer、template、最大长度、packing/mask 语义、各 role token 数和无监督样本率。训练后用推理端同一模板做 golden prompt 回归；模板不一致会比很多超参数错误更致命。

In [ ]:
example_payload112=json.dumps([m.__dict__ for m in conv112],ensure_ascii=False,sort_keys=True); example_hash112=hashlib.sha256(example_payload112.encode()).hexdigest()
manifest112={"schema":1,"dataset":"sft-v4","split_key":"conversation_family","tokenizer":"word-demo-v1","template":"roles+eos-v1","assistant_only":True,"ignore_index":IGNORE112,"packing":"block_causal","max_length":2048}; digest112=hashlib.sha256(json.dumps(manifest112,sort_keys=True).encode()).hexdigest()
assert len(example_hash112)==64 and len(digest112)==64
assert manifest112["assistant_only"] and manifest112["ignore_index"]==-100
assert manifest112["packing"]=="block_causal"

## 面试总结

强回答路径是：**对话 schema → train-only tokenizer/固定 special ID → chat template → assistant-only mask → causal shift → block-diagonal packing/position → 完整 turn 截断 → 最小模型梯度 oracle → 数据与模板 manifest**。SFT 数据管线的一位错位，就可能让昂贵训练学到错误目标。

延伸阅读：[Hugging Face Chat Templates](https://huggingface.co/docs/transformers/chat_templating)、[InstructGPT](https://arxiv.org/abs/2203.02155)、[Megatron-LM](https://github.com/NVIDIA/Megatron-LM)。